# Miro Face Detector - CNN Model Training

This notebook trains a Convolutional Neural Network (CNN) to classify images as either 'Miro' (1) or 'Random' (0). This version includes stability fixes to address validation loss spikes and overfitting:
1. **Lower Learning Rate**: Initial LR reduced to 0.0001 to prevent overshooting.
2. **Learning Rate Scheduler**: Automatically scales down LR on plateaus.
3. **Early Stopping**: Halts training and restores the best weights if validation performance degrades.

## Configuration

For maximum speed in Google Colab, package your processed dataset into a zip file named `processed.zip` and place it in your drive at `/content/drive/MyDrive/processed.zip`. The script will automatically pull it to local storage and extract it.

Alternatively, ensure your `config.yaml` file is properly positioned before training. For example, if the location of the dataset is at `/content/drive/MyDrive/processed`, place the config file there or at the project root.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
import matplotlib.pyplot as plt
import zipfile
import shutil

colab_zip_path = '/content/drive/MyDrive/processed.zip'
local_extract_dir = '/content/local_data'
yaml_path = '../config.yaml'

if os.path.exists(colab_zip_path):
    print("Found processed.zip. Copying and extracting to local runtime for faster access...")
    local_zip = '/content/processed.zip'
    if not os.path.exists(local_zip):
        shutil.copy2(colab_zip_path, local_zip)
    if not os.path.exists(local_extract_dir):
        os.makedirs(local_extract_dir)
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall(local_extract_dir)

    possible_csv = os.path.join(local_extract_dir, 'dataset.csv')
    possible_csv_nested = os.path.join(local_extract_dir, 'processed', 'dataset.csv')

    if os.path.exists(possible_csv_nested):
        DATA_CSV = possible_csv_nested
        DATA_DIR = os.path.join(local_extract_dir, 'processed')
    else:
        DATA_CSV = possible_csv
        DATA_DIR = local_extract_dir

    IMG_SIZE = 128
    MODEL_OUTPUT = '/content/drive/MyDrive/models/miro_detector.keras'
    MAX_POSITIVE = -1
    MAX_NEGATIVE = -1
elif os.path.exists(yaml_path):
    import yaml
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    DATA_CSV = '../' + config['data']['dataset_csv']
    IMG_SIZE = config['model']['img_size']
    MODEL_OUTPUT = '../' + config['model']['output_path']
    DATA_DIR = '../'
    MAX_POSITIVE = config.get('data', {}).get('max_positive', -1)
    MAX_NEGATIVE = config.get('data', {}).get('max_negative', -1)
else:
    # Fallback for Colab execution where structure might differ
    DATA_CSV = '/content/drive/MyDrive/processed/dataset.csv'
    IMG_SIZE = 128
    MODEL_OUTPUT = '/content/drive/MyDrive/models/miro_detector.keras'
    DATA_DIR = '/content/drive/MyDrive/'
    MAX_POSITIVE = -1
    MAX_NEGATIVE = -1

# Ensure we use the modern keras extension to avoid warnings
if MODEL_OUTPUT.endswith('.h5'):
    MODEL_OUTPUT = MODEL_OUTPUT[:-3] + '.keras'

print(f"TensorFlow Version: {tf.__version__}")
print(f"Using DATA_DIR: {DATA_DIR}")
print(f"Using DATA_CSV: {DATA_CSV}")

Found processed.zip. Copying and extracting to local runtime for faster access...
TensorFlow Version: 2.19.0
Using DATA_DIR: /content/local_data/processed
Using DATA_CSV: /content/local_data/processed/dataset.csv


## 1. Data Loading and Preparation

In [3]:
df = pd.read_csv(DATA_CSV)

# Apply frame limits if specified
pos_df = df[df['label'] == 1]
neg_df = df[df['label'] == 0]

if MAX_POSITIVE != -1 and len(pos_df) > MAX_POSITIVE:
    print(f"Limiting positive frames from {len(pos_df)} to {MAX_POSITIVE}")
    pos_df = pos_df.sample(n=MAX_POSITIVE, random_state=42)

if MAX_NEGATIVE != -1 and len(neg_df) > MAX_NEGATIVE:
    print(f"Limiting negative frames from {len(neg_df)} to {MAX_NEGATIVE}")
    neg_df = neg_df.sample(n=MAX_NEGATIVE, random_state=42)

df = pd.concat([pos_df, neg_df]).sample(frac=1, random_state=42).reset_index(drop=True)

def resolve_paths(dataframe):
    valid_paths = []
    valid_labels = []
    for path, label in zip(dataframe['filepath'], dataframe['label']):
        filepath = str(path).replace(chr(92), '/')
        img_path = os.path.join(DATA_DIR, filepath)

        if not os.path.exists(img_path) and 'data/processed/' in filepath:
            local_alt = os.path.join(DATA_DIR, filepath.split('data/processed/')[-1])
            drive_alt = os.path.join('/content/drive/MyDrive/processed/', filepath.split('data/processed/')[-1])
            if os.path.exists(local_alt):
                img_path = local_alt
            elif os.path.exists(drive_alt):
                img_path = drive_alt

        if os.path.exists(img_path):
            valid_paths.append(img_path)
            valid_labels.append(label)

    return valid_paths, valid_labels

def process_path(file_path, label):
    # Read and decode the image directly from disk
    img = tf.io.read_file(file_path)
    # Use decode_jpeg or decode_png depending on your dataset format
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = img / 255.0
    return img, label

def create_dataset(dataframe, batch_size=32):
    paths, labels = resolve_paths(dataframe)

    if len(paths) == 0:
        raise ValueError("No images found. Please verify DATA_DIR and filepath alignment.")

    # Create the dataset stream
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    # Map the processing function across multiple CPU cores
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    # Batch the data and prefetch the next batch to the GPU
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

BATCH_SIZE = 32

print("Creating training dataset pipeline...")
train_dataset = create_dataset(train_df, batch_size=BATCH_SIZE)

print("Creating testing dataset pipeline...")
test_dataset = create_dataset(test_df, batch_size=BATCH_SIZE)

Loading training data...
Loaded 10 out of 15617 images...
Loaded 20 out of 15617 images...
Loaded 30 out of 15617 images...
Loaded 40 out of 15617 images...
Loaded 50 out of 15617 images...
Loaded 60 out of 15617 images...
Loaded 70 out of 15617 images...
Loaded 80 out of 15617 images...
Loaded 90 out of 15617 images...
Loaded 100 out of 15617 images...
Loaded 110 out of 15617 images...
Loaded 120 out of 15617 images...
Loaded 130 out of 15617 images...
Loaded 140 out of 15617 images...
Loaded 150 out of 15617 images...
Loaded 160 out of 15617 images...
Loaded 170 out of 15617 images...
Loaded 180 out of 15617 images...
Loaded 190 out of 15617 images...
Loaded 200 out of 15617 images...
Loaded 210 out of 15617 images...
Loaded 220 out of 15617 images...
Loaded 230 out of 15617 images...
Loaded 240 out of 15617 images...
Loaded 250 out of 15617 images...
Loaded 260 out of 15617 images...
Loaded 270 out of 15617 images...
Error loading image: /content/local_data/processed/data/processed/

KeyboardInterrupt: 

## 2. Model Architecture

Building a deeper VGG-style CNN to learn highly complex facial features. Layers are grouped into blocks with Batch Normalization.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # Data Augmentation
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 4
    layers.Conv2D(256, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu', padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# Use a lower learning rate for better stability
optimizer = optimizers.Adam(learning_rate=0.0001)

model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

## 3. Training and Evaluation

In [ ]:
# Implement callbacks to mitigate overfitting and spikes
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=6, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1
)

history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=test_dataset,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')
plt.show()

## 4. Exporting Model Weights

In [ ]:
os.makedirs(os.path.dirname(MODEL_OUTPUT), exist_ok=True)
model.save(MODEL_OUTPUT)
print(f"Model saved to {MODEL_OUTPUT}")